# Adversarial TTS - Class-Based Architecture (Google Colab)

This notebook uses the refactored class-based architecture:
- **EnvironmentLoader**: Handles model loading and environment setup
- **AdversarialTrainer**: Runs the optimization loop (returns results)
- **RunLogger**: Handles all output and logging (called separately)

This separation allows you to re-run logging/visualization without re-running optimization.

## Setup
Run the cell below to clone the repository and install dependencies.

In [ ]:
%%bash
git clone https://github.com/Vorgesetzter/StyleTTS2
cd StyleTTS2
pip install -r requirements.txt
sudo apt-get install espeak-ng
git-lfs clone https://huggingface.co/yl4579/StyleTTS2-LJSpeech
mkdir -p Audio
mv StyleTTS2-LJSpeech/Models/* Audio/
rm -rf StyleTTS2-LJSpeech

## 1. Configuration & Imports

In [ ]:
%cd StyleTTS2

import torch
import nltk
nltk.download('punkt_tab')

# Import class-based modules
from Trainer.ModelLoader import EnvironmentLoader
from Trainer.AdversarialTrainer import AdversarialTrainer
from Trainer.RunLogger import RunLogger


class NotebookArgs:
    """Configuration class mimicking argparse for notebook usage."""
    def __init__(self):
        # String parameters
        self.ground_truth_text = "I think the NFL is lame and boring"
        self.target_text = "The Seattle Seahawks are the best Team in the world"

        # Numeric parameters
        self.loop_count = 1
        self.num_generations = 100
        self.pop_size = 200
        self.iv_scalar = 0.5
        self.size_per_phoneme = 1
        self.batch_size = 100

        # Boolean parameters
        self.notify = False
        self.subspace_optimization = False
        self.multi_gpu = True  # Auto-enabled for Colab with multiple GPUs

        # Enum/Selection parameters
        self.mode = "TARGETED"
        self.ACTIVE_OBJECTIVES = ["PESQ", "WHISPER_PROB"]
        self.thresholds = ["PESQ=0.2", "WHISPER_PROB=0.6"]


args = NotebookArgs()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
print(f"Available GPUs: {torch.cuda.device_count()}")

## 2. Initialize Environment

Multi-GPU is automatically configured during initialization if `multi_gpu=True`.

In [ ]:
# Initialize environment using EnvironmentLoader
# Multi-GPU wrapping happens automatically based on args.multi_gpu
loader = EnvironmentLoader(args, device)
config, models, audio, embeds, objective_manager = loader.initialize()

if config is None:
    raise RuntimeError("Initialization failed.")

print("\nEnvironment initialized successfully!")

## 3. Run Optimization

The trainer only runs optimization and returns results. Logging is done separately.

In [ ]:
# Create trainer (optimization only - no logging)
trainer = AdversarialTrainer(
    config=config,
    models=models,
    audio=audio,
    embeds=embeds,
    objective_manager=objective_manager,
    device=device
)

# Run optimization - returns list of OptimizationResult
results = trainer.run()

print(f"\nOptimization complete! Got {len(results)} result(s).")

## 4. Log Results (Separate Step)

You can re-run this cell with different logging settings without re-running optimization.

In [ ]:
# Create logger and save results
logger = RunLogger(config, models, audio, device)

for i, result in enumerate(results):
    print(f"Saving results for loop {i + 1}...")
    logger.finalize_run(
        result.fitness_data,
        result.generation_count,
        result.elapsed_time
    )

print(f"\nResults saved to: {logger.folder_path}")

## 5. Re-generate Visualizations (Optional)

You can regenerate graphs with different settings without re-running optimization.

In [ ]:
# Example: Regenerate visualizations with the existing results
from Trainer.GraphPlotter import GraphPlotter

# Use the last result's fitness data
if results:
    last_result = results[-1]
    
    # Create a new plotter (you could customize settings here)
    plotter = GraphPlotter(
        folder_path=logger.folder_path,
        active_objectives=config.active_objectives,
        num_generations=config.num_generations
    )
    
    # Regenerate all visualizations
    plotter.generate_all_visualizations(last_result.fitness_data)
    print("Visualizations regenerated!")

## 6. Download Results (Optional)

In [ ]:
extract = False  # Set to True to download results

if extract:
    !zip -r colab_results.zip /content/StyleTTS2/output/

    from google.colab import files
    files.download('colab_results.zip')